# Red neuronal desde cero · Tema 2 · Sesión 2

**Máster en Ingeniería de Automatización con IA Agéntica · EBIS**
Autor: Manuel Díaz Bendito

En este notebook construimos una red neuronal **desde cero**, entendiendo cada pieza: qué es una neurona, cómo se propaga la señal hacia adelante (*forward*), cómo se mide el error y cómo aprende mediante **descenso de gradiente**.

Empezamos con la unidad mínima —**una sola neurona**, que equivale a una regresión logística— y la entrenamos para distinguir dígitos manuscritos del dataset **MNIST**. Después daremos el salto a una red **multicapa (MLP)** y veremos por qué añadir capas ocultas permite resolver problemas que una sola neurona no puede.

Usamos **PyTorch**, el framework estándar de la industria, pero escribimos el bucle de entrenamiento a mano para ver explícitamente las cuatro fases del aprendizaje: *forward → loss → backward → update*.

## 0 · Compatibilidad de hardware (léeme primero)

Este notebook funciona en **cualquier ordenador**. El código detecta automáticamente el mejor dispositivo de cómputo disponible y se adapta a él:

- **CUDA** → GPU NVIDIA (Windows / Linux). El más rápido para entrenar.
- **MPS** → GPUs de Apple Silicon (M1/M2/M3/M4). Acelera mucho en Mac.
- **CPU** → cualquier máquina. Funciona siempre; más lento, pero perfectamente válido para los modelos de este notebook.

No tienes que cambiar nada manualmente: la celda de detección de dispositivo elige sola. Si entrenas en CPU, los tiempos de este notebook siguen siendo de pocos minutos.

## 1 · Instalación e importación de librerías

Solo necesitamos PyTorch, torchvision (para descargar MNIST), NumPy y Matplotlib. Si ya los tienes instalados, salta la primera celda.

In [ ]:
# Ejecuta una sola vez si te falta alguna librería:
# %pip install torch torchvision numpy matplotlib scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torchvision import datasets, transforms

# Semilla para reproducibilidad: mismos resultados en cada ejecución
torch.manual_seed(42)
np.random.seed(42)
print("PyTorch:", torch.__version__)

## 2 · Detección automática de dispositivo

Esta celda hace el notebook universal: prueba CUDA → MPS → CPU y elige el mejor disponible.

In [ ]:
def elegir_dispositivo():
    """Devuelve el mejor dispositivo disponible: CUDA > MPS > CPU."""
    if torch.cuda.is_available():
        print(f"✅ Usando CUDA (GPU NVIDIA): {torch.cuda.get_device_name(0)}")
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        print("✅ Usando MPS (Apple Silicon)")
        return torch.device("mps")
    print("⚠️  No se ha detectado GPU. Usando CPU (funciona igual, algo más lento).")
    return torch.device("cpu")

device = elegir_dispositivo()
print("Dispositivo seleccionado:", device)

## 3 · La intuición: ¿qué es una neurona?

Una **neurona artificial** hace tres cosas:

1. **Combina** sus entradas $x_1, x_2, \dots, x_n$ con unos **pesos** $w_1, \dots, w_n$ y un **sesgo** (*bias*) $b$:
$$z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b = \mathbf{w}^\top \mathbf{x} + b$$

2. **Activa** ese resultado con una función no lineal. Usamos la **sigmoide**, que aplasta cualquier número al rango $(0, 1)$ y nos da una probabilidad:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

3. **Decide**: si $\sigma(z) > 0.5$, predice la clase 1; si no, la clase 0.

Una sola neurona con activación sigmoide es **exactamente** una regresión logística. Vamos a verla por dentro.

In [ ]:
# La función de activación sigmoide
def sigmoide(z):
    return 1 / (1 + torch.exp(-z))

z = torch.linspace(-10, 10, 200)
plt.figure(figsize=(7, 3.5))
plt.plot(z.numpy(), sigmoide(z).numpy(), color="#08627F", linewidth=2.5)
plt.axhline(0.5, color="grey", linestyle="--", linewidth=1)
plt.axvline(0, color="grey", linestyle="--", linewidth=1)
plt.title("Función sigmoide: convierte cualquier número en una probabilidad (0-1)")
plt.xlabel("z = w·x + b"); plt.ylabel("σ(z)")
plt.grid(alpha=0.3); plt.show()

## 4 · Cargamos los datos · MNIST (dígitos 0 y 1)

**MNIST** es el "hola mundo" de la visión por computador: 70.000 imágenes de dígitos manuscritos de 28×28 píxeles en escala de grises. Empezamos con un problema **binario sencillo**: distinguir **ceros de unos**.

In [ ]:
# Descargamos MNIST completo (se cachea localmente tras la primera vez)
transform = transforms.ToTensor()  # convierte imágenes a tensores con valores en [0, 1]
mnist_train = datasets.MNIST(root="./datos", train=True, download=True, transform=transform)
mnist_test  = datasets.MNIST(root="./datos", train=False, download=True, transform=transform)

# Pasamos todo a tensores en memoria (MNIST es pequeño, cabe de sobra)
X_train_full = mnist_train.data.float() / 255.0   # (60000, 28, 28), normalizado a [0,1]
y_train_full = mnist_train.targets
X_test_full  = mnist_test.data.float() / 255.0
y_test_full  = mnist_test.targets
print("Train:", X_train_full.shape, "| Test:", X_test_full.shape)

In [ ]:
def filtrar_dos_clases(X, y, clase0, clase1):
    """Devuelve solo las imágenes de dos clases, con etiquetas 0/1."""
    mascara = (y == clase0) | (y == clase1)
    X_sel = X[mascara]
    y_sel = (y[mascara] == clase1).long()   # clase0 -> 0, clase1 -> 1
    return X_sel, y_sel

CLASE_0, CLASE_1 = 0, 1
X_train, y_train = filtrar_dos_clases(X_train_full, y_train_full, CLASE_0, CLASE_1)
X_test,  y_test  = filtrar_dos_clases(X_test_full,  y_test_full,  CLASE_0, CLASE_1)
print(f"Distinguiendo {CLASE_0} vs {CLASE_1}")
print(f"Train: {len(X_train)} imágenes | Test: {len(X_test)} imágenes")

In [ ]:
def mostrar_ejemplos(X, y, n=9):
    idx = np.random.choice(len(X), n, replace=False)
    fig, axes = plt.subplots(3, 3, figsize=(5, 5))
    for ax, i in zip(axes.flatten(), idx):
        ax.imshow(X[i].numpy(), cmap="gray")
        ax.set_title(f"Etiqueta: {int(y[i])}"); ax.axis("off")
    plt.tight_layout(); plt.show()

mostrar_ejemplos(X_train, y_train)

## 5 · Preparamos los datos para la red

Una neurona recibe un **vector** de números, no una matriz 2D. "Aplanamos" cada imagen de 28×28 en un vector de **784 valores** (uno por píxel). Cada píxel es una entrada $x_i$ de la neurona.

Movemos también los datos al dispositivo elegido.

In [ ]:
# Aplanamos: (N, 28, 28) -> (N, 784)
X_train_flat = X_train.reshape(len(X_train), -1).to(device)
X_test_flat  = X_test.reshape(len(X_test),  -1).to(device)
y_train_dev  = y_train.float().to(device)   # float para la BCE loss
y_test_dev   = y_test.float().to(device)

n_features = X_train_flat.shape[1]
print(f"Cada imagen es ahora un vector de {n_features} valores (28×28 píxeles)")
print("Shape de entrenamiento:", X_train_flat.shape)

## 6 · Una neurona desde cero · las 4 fases del aprendizaje

Definimos a mano los parámetros (`w` y `b`) y el bucle de entrenamiento. En cada iteración (*época*) repetimos:

1. **Forward**: calculamos la predicción $\hat{y} = \sigma(\mathbf{X}\mathbf{w} + b)$.
2. **Loss**: medimos el error con la **entropía cruzada binaria** (penaliza estar seguro y equivocado):
$$\mathcal{L} = -\frac{1}{m}\sum_i \big[ y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i) \big]$$
3. **Backward**: PyTorch calcula automáticamente los gradientes $\frac{\partial \mathcal{L}}{\partial w}$ con `loss.backward()`.
4. **Update**: movemos los pesos en la dirección que reduce el error: $w \leftarrow w - \eta \frac{\partial \mathcal{L}}{\partial w}$.

`requires_grad=True` le dice a PyTorch que vigile esos tensores para poder derivarlos.

In [ ]:
def entrenar_neurona(X, y, n_iter=200, lr=0.5):
    m, n = X.shape
    # Parámetros entrenables, inicializados a cero
    w = torch.zeros((n, 1), device=device, requires_grad=True)
    b = torch.zeros(1, device=device, requires_grad=True)
    costes = []

    for it in range(n_iter):
        # 1) FORWARD
        z = X @ w + b                       # combinación lineal
        y_hat = torch.sigmoid(z).squeeze()  # activación -> probabilidad
        # 2) LOSS (entropía cruzada binaria), con epsilon para estabilidad numérica
        eps = 1e-7
        loss = -torch.mean(y * torch.log(y_hat + eps) + (1 - y) * torch.log(1 - y_hat + eps))
        # 3) BACKWARD: gradientes automáticos
        loss.backward()
        # 4) UPDATE: descenso de gradiente (sin construir el grafo de nuevo)
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
            w.grad.zero_(); b.grad.zero_()   # reseteamos los gradientes
        costes.append(loss.item())
        if it % 40 == 0 or it == n_iter - 1:
            print(f"iter {it:4d} | loss {loss.item():.4f}")
    return w.detach(), b.detach(), costes

w, b, costes = entrenar_neurona(X_train_flat, y_train_dev, n_iter=200, lr=0.5)

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(costes, color="#08627F", linewidth=2)
plt.title("La loss baja: la neurona está aprendiendo")
plt.xlabel("Iteración"); plt.ylabel("Loss (entropía cruzada)")
plt.grid(alpha=0.3); plt.show()

## 7 · Evaluamos la neurona

Medimos el **accuracy** (porcentaje de aciertos) en train y test, y dibujamos la **matriz de confusión** para ver dónde se equivoca.

In [ ]:
def predecir(X, w, b):
    with torch.no_grad():
        return (torch.sigmoid(X @ w + b).squeeze() > 0.5).long()

def accuracy(X, y, w, b):
    return (predecir(X, w, b) == y.long()).float().mean().item()

print(f"Accuracy train: {accuracy(X_train_flat, y_train_dev, w, b)*100:.2f} %")
print(f"Accuracy test : {accuracy(X_test_flat,  y_test_dev,  w, b)*100:.2f} %")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred = predecir(X_test_flat, w, b).cpu().numpy()
cm = confusion_matrix(y_test.numpy(), y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=[str(CLASE_0), str(CLASE_1)])
disp.plot(cmap="Blues", colorbar=False)
plt.title(f"Matriz de confusión · {CLASE_0} vs {CLASE_1}"); plt.show()

## 8 · Un problema más difícil · 5 vs 6

Los ceros y unos son fáciles de separar. Probemos un par más parecido visualmente, **5 vs 6**, para ver hasta dónde llega una sola neurona.

In [ ]:
X5, y5 = filtrar_dos_clases(X_train_full, y_train_full, 5, 6)
X5_test, y5_test = filtrar_dos_clases(X_test_full, y_test_full, 5, 6)
X5f = X5.reshape(len(X5), -1).to(device); y5d = y5.float().to(device)
X5tf = X5_test.reshape(len(X5_test), -1).to(device); y5td = y5_test.float().to(device)

w5, b5, _ = entrenar_neurona(X5f, y5d, n_iter=200, lr=0.5)
print(f"\nAccuracy test (5 vs 6) con UNA neurona: {accuracy(X5tf, y5td, w5, b5)*100:.2f} %")

## 9 · El salto: una red multicapa (MLP)

Una sola neurona solo puede trazar una **frontera lineal**. Para capturar patrones más complejos apilamos **capas de neuronas** con activaciones no lineales en medio: es el **Perceptrón Multicapa (MLP)**.

Aquí usamos la API idiomática de PyTorch (`nn.Module`): definimos las capas y PyTorch se encarga del forward y de los gradientes. Nuestra red tiene una **capa oculta de 128 neuronas** con activación ReLU.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_entrada, n_oculta=128):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(n_entrada, n_oculta),   # capa oculta
            nn.ReLU(),                        # no linealidad
            nn.Linear(n_oculta, 1),           # capa de salida (1 logit)
        )
    def forward(self, x):
        return self.red(x).squeeze(1)

def entrenar_mlp(X, y, n_iter=200, lr=0.05):
    modelo = MLP(X.shape[1]).to(device)
    optimizador = torch.optim.Adam(modelo.parameters(), lr=lr)
    criterio = nn.BCEWithLogitsLoss()   # combina sigmoide + entropía cruzada (estable)
    costes = []
    for it in range(n_iter):
        logits = modelo(X)
        loss = criterio(logits, y)
        optimizador.zero_grad()
        loss.backward()
        optimizador.step()
        costes.append(loss.item())
        if it % 40 == 0 or it == n_iter - 1:
            print(f"iter {it:4d} | loss {loss.item():.4f}")
    return modelo, costes

modelo_mlp, costes_mlp = entrenar_mlp(X5f, y5d, n_iter=200, lr=0.05)

In [ ]:
def accuracy_mlp(modelo, X, y):
    modelo.eval()
    with torch.no_grad():
        pred = (torch.sigmoid(modelo(X)) > 0.5).long()
    return (pred == y.long()).float().mean().item()

print(f"Accuracy test (5 vs 6):")
print(f"  · Una neurona      : {accuracy(X5tf, y5td, w5, b5)*100:.2f} %")
print(f"  · MLP (128 ocultas): {accuracy_mlp(modelo_mlp, X5tf, y5td)*100:.2f} %")

## 10 · Lo que has aprendido

- Una **neurona** = combinación lineal + activación sigmoide = regresión logística.
- El aprendizaje son **4 fases** que se repiten: forward → loss → backward → update.
- El **descenso de gradiente** ajusta los pesos para minimizar el error.
- Apilar capas (**MLP**) permite aprender fronteras no lineales y mejora en problemas difíciles.

Todo esto —neuronas, activaciones, loss, backprop, descenso de gradiente— es **exactamente** lo que hay dentro de las redes más grandes. La diferencia con un modelo de visión moderno o un GPT es de **escala y arquitectura**, no de naturaleza.

> En la siguiente sesión veremos las **redes convolucionales**, que explotan la estructura espacial de las imágenes en lugar de aplanarlas en un vector.